# Azimuthal sector emission profiles

Explore a radial wind at one signed speed. The dashed curve uses the same wind and viewing angles with full azimuthal coverage. Both curves use a 6.7 keV intrinsic line and unit intrinsic all-sky photon flux. Their areas show the emitting solid angle and the Doppler weight `δ³`.

The blue surface is the specified sector. With **bicone** enabled, the purple surface is its antipodal copy. The arrow points toward the observer. **phi = 0°** lies along that arrow's equatorial projection.

Run all cells in Jupyter with NumPy, SciPy, Matplotlib, ipywidgets and ipympl installed.

In [9]:
%matplotlib widget
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import Checkbox, FloatSlider, HBox, HTML, VBox

for root in (Path.cwd(), *Path.cwd().parents):
    if (root / "src/redwine/experimental/kernels.py").is_file():
        sys.path.insert(0, str(root / "src"))
        break

from redwine.experimental.kernels import Sector, line

E0 = 6.7  # keV
norm = 1.0  # intrinsic all-sky photon flux

In [10]:
def slider(name, value, low, high, step, fmt=".0f"):
    return FloatSlider(description=name, value=value, min=low, max=high, step=step,
                       readout_format=fmt, continuous_update=False)


vout = slider("vout (c)", 0.15, 0.005, 0.3, .005, ".3f")
tin = slider("tin (°)", 20, 0, 50, 1)
tout = slider("tout (°)", 50, 20, 90, 1)
incl = slider("incl (°)", 45, 0, 90, 1)
phi = slider("phi (°)", 350, 0, 360, 1)
dphi = slider("dphi (°)", 80, 0, 360, 1)
bicone = Checkbox(description="bicone", value=False)
summary = HTML()

with plt.ioff():
    fig = plt.figure(figsize=(10, 4.5), layout="constrained")
    ax = fig.add_subplot(121)
    geometry = fig.add_subplot(122, projection="3d")
    sector_curve, = ax.plot([], [], color="tab:blue", lw=1.6, label="Sector")
    full_curve, = ax.plot([], [], "--", color="0.4", lw=1.2, label="Full azimuth")
    ax.axvline(E0, color="0.7", lw=1)
    ax.set(xlabel="Energy (keV)", ylabel="Photon flux density (keV⁻¹)")
    ax.legend(loc="upper right")


def draw_geometry(wind):
    geometry.clear()
    theta = np.deg2rad(np.linspace(wind.tin, wind.tout, 17))[:, None]
    azimuth = np.deg2rad(wind.phi + np.linspace(-.5, .5, 65)*wind.dphi)
    x = np.sin(theta)*np.cos(azimuth)
    y = np.sin(theta)*np.sin(azimuth)
    z = np.broadcast_to(np.cos(theta), x.shape)
    equator = np.linspace(0, 2*np.pi, 100)
    geometry.plot(np.cos(equator), np.sin(equator), np.zeros_like(equator),
                  color="0.75", lw=1)
    geometry.plot([0, 0], [0, 0], [-1.1, 1.1], color="0.65", lw=1)
    if wind.solid_angle_fraction > 0:
        geometry.plot_surface(x, y, z, color="tab:blue", alpha=.75)
        if wind.bicone:
            geometry.plot_surface(-x, -y, -z, color="tab:purple", alpha=.65)
    angle = np.deg2rad(wind.incl)
    geometry.quiver(0, 0, 0, np.sin(angle), 0, np.cos(angle),
                    length=1.25, color="tab:orange", arrow_length_ratio=.16)
    geometry.set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), zlim=(-1.3, 1.3),
                 xlabel="x", ylabel="y", zlabel="Wind axis", title="Sector and sight line")
    geometry.set_box_aspect((1, 1, 1))
    geometry.set(xticks=[], yticks=[], zticks=[])


def update():
    wind = Sector(vout.value, tin.value, tout.value, incl.value,
                  phi.value, dphi.value, bicone.value)
    half_span = max(.008, 1.03*np.arctanh(abs(wind.vout)))
    edges = E0*np.exp(np.linspace(-half_span, half_span, 1602))
    centers = (edges[:-1]+edges[1:])/2
    widths = np.diff(edges)
    sector_flux = norm*line(edges, E0, wind)
    full_flux = norm*line(edges, E0, replace(wind, dphi=360))
    for curve, flux in ((sector_curve, sector_flux), (full_curve, full_flux)):
        curve.set_data(centers, flux/widths)
        curve.set_drawstyle("steps-mid")
    peak = max((sector_flux/widths).max(), (full_flux/widths).max())
    # ax.set(xlim=(edges[0], edges[-1]), ylim=(0, 1.12*peak if peak > 0 else 1))
    ax.set(xlim=(5.5, 8), ylim=(0, 0.8))
    total = sector_flux.sum()
    centroid = f"{sector_flux @ centers / total:.4f} keV" if total > 0 else "—"
    summary.value = (f"<b>Sector</b> {total:.6f} &nbsp; "
                     f"<b>Full azimuth</b> {full_flux.sum():.6f} &nbsp; "
                     f"<b>Sector centroid</b> {centroid}")
    draw_geometry(wind)
    fig.canvas.draw_idle()


syncing = False


def sync_geometry(change):
    global syncing
    if syncing:
        return
    syncing = True
    try:
        # Clamp the inner edge first when the bicone makes the allowed cap smaller.
        outer_limit = 90 if bicone.value else 180
        tin.max = min(tout.value, outer_limit)
        tout.min = tin.value
        tout.max = outer_limit
    finally:
        syncing = False
    update()


for control in (tin, tout, bicone):
    control.observe(sync_geometry, names="value")
for control in (vout, incl, phi, dphi):
    control.observe(lambda change: update(), names="value")

update()
display(VBox([HBox([vout, incl, bicone]), HBox([tin, tout]),
              HBox([phi, dphi]), summary, fig.canvas]))